# Künstliche Intelligenz für unseren Online-Shop (Einfach erklärt)

Wir wollen vorhersehen: **Kauft der Besucher auf unserer Webseite etwas (Ja/Nein)?**
Wenn wir das vorher wissen, können wir den Leuten, die noch unentschlossen sind, z.B. gezielt einen Rabattcode anzeigen.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# 1. Wir erfinden 1.500 Test-Besucher für unseren Shop
np.random.seed(42)
n_samples = 1500

# Was wissen wir über die Besucher?
time_on_site = np.clip(np.random.normal(5, 3, n_samples), 0.5, 30) # Zeit auf der Seite in Minuten
pages_visited = np.random.poisson(4, n_samples) # Anzahl der besuchten Seiten
added_to_cart = np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3]) # 1 = Hat etwas im Warenkorb
discount_offered = np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]) # 1 = Hat einen Rabatt gesehen

# Wir legen fest: Wer länger bleibt oder den Warenkorb nutzt, kauft mit höherer Wahrscheinlichkeit
logit = -3 + 0.3*time_on_site + 0.5*pages_visited + 2.5*added_to_cart + 1.2*discount_offered
buy_probability = 1 / (1 + np.exp(-logit))
purchased = (np.random.rand(n_samples) < buy_probability).astype(int) # 1 = Hat am Ende gekauft

# Alle Daten in eine übersichtliche Tabelle packen
df = pd.DataFrame({
    'Minuten_auf_Seite': time_on_site,
    'Seiten_besucht': pages_visited,
    'Warenkorb_gefuellt': added_to_cart,
    'Rabatt_gesehen': discount_offered,
    'HAT_GEKAUFT': purchased
})

print("So sehen unsere Besucher-Daten aus (erste 5 Personen):")
display(df.head())

So sehen unsere Besucher-Daten aus (erste 5 Personen):


,Minuten_auf_Seite,Seiten_besucht,Warenkorb_gefuellt,Rabatt_gesehen,HAT_GEKAUFT
0,6.490142,3,0,0,1
1,4.585207,2,1,0,0
2,6.943066,5,0,1,1
3,9.569090,2,0,0,1
4,4.297540,8,0,0,1


### Daten anpassen (Skalieren)
Eine KI kann nicht gut vergleichen, wenn eine Zahl '30' (Minuten) und die andere '1' (Warenkorb Ja/Nein) ist. Daher rechnen wir alle Werte so um, dass sie für den Computer gleich groß aussehen.

In [6]:
# 2. Daten aufteilen und anpassen
# Wir nehmen 80% der Daten zum Lernen (Training) und 20% zum Abfragen (Test)
X = df[['Minuten_auf_Seite', 'Seiten_besucht', 'Warenkorb_gefuellt', 'Rabatt_gesehen']]
y = df['HAT_GEKAUFT']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Skalieren (Zahlen für die KI 'glätten')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"{X_train_scaled.shape[0]} Besucher werden genutzt, damit die KI lernt.")
print(f"{X_test_scaled.shape[0]} Besucher heben wir auf, um die KI am Ende zu testen.")

1200 Besucher werden genutzt, damit die KI lernt.
300 Besucher heben wir auf, um die KI am Ende zu testen.


### Das künstliche Gehirn (Neuronales Netz) bauen
Wir nutzen ein fertiges Programm (Scikit-Learn). Wir sagen der KI: 'Schau dir die 1.200 Trainings-Besucher an und lerne die Muster, wer am Ende kauft und wer nicht.'

In [7]:
# 3. KI trainieren (Lernen lassen)
mlp = MLPClassifier(
    hidden_layer_sizes=(8, 4), # Wie groß das 'Gehirn' ist
    activation='relu',
    max_iter=1000, # Er darf das Material bis zu 1000 Mal durchgehen
    random_state=42
)

print("Die KI lernt jetzt aus den Daten... Bitte warten...")
mlp.fit(X_train_scaled, y_train)
print("Fertig! Die KI hat die Muster gelernt.")

Die KI lernt jetzt aus den Daten... Bitte warten...
Fertig! Die KI hat die Muster gelernt.


### Die Prüfung: Wie gut ist die KI?
Jetzt zeigen wir der KI die restlichen 300 Besucher (die Testdaten). Wir kennen das Ergebnis schon, aber die KI muss nun raten. Dann vergleichen wir ihre Vorhersage mit der Wahrheit.

In [8]:
# 4. Prüfung der Vorhersagen
from sklearn.metrics import confusion_matrix

vorhersagen = mlp.predict(X_test_scaled)
matrix = confusion_matrix(y_test, vorhersagen)
wahr_nein, falsch_ja, falsch_nein, wahr_ja = matrix.ravel()

print("=== ZEUGNIS FÜR DIE KI (Verständliche Auswertung) ===\n")
print(f"Wir haben {len(y_test)} Besucher getestet.\n")

print("HIER HAT DIE KI RECHT GEHABT (Die guten Zahlen):")
print(f"✔️ {wahr_nein} Besucher haben nicht gekauft. Die KI wusste das.")
print(f"✔️ {wahr_ja} Besucher haben gekauft. Die KI hat es korrekt vorhergesehen.\n")

print("HIER HAT SICH DIE KI GEIRRT (Die schlechten Zahlen):")
print(f"❌ {falsch_ja} Fehlalarme: Die KI dachte, sie kaufen. Haben sie aber NICHT.")
print(f"❌ {falsch_nein} Übersehen: Sie haben gekauft, aber die KI dachte, sie tun es NICHT.\n")

genauigkeit = ((wahr_nein + wahr_ja) / len(y_test)) * 100
print("=== ZUSAMMENFASSUNG ===")
print(f"Die KI lag in {genauigkeit:.1f} % der Fälle richtig!")

=== ZEUGNIS FÜR DIE KI (Verständliche Auswertung) ===

Wir haben 300 Besucher getestet.

HIER HAT DIE KI RECHT GEHABT (Die guten Zahlen):
✔️ 36 Besucher haben nicht gekauft. Die KI wusste das.
✔️ 203 Besucher haben gekauft. Die KI hat es korrekt vorhergesehen.

HIER HAT SICH DIE KI GEIRRT (Die schlechten Zahlen):
❌ 37 Fehlalarme: Die KI dachte, sie kaufen. Haben sie aber NICHT.
❌ 24 Übersehen: Sie haben gekauft, aber die KI dachte, sie tun es NICHT.

=== ZUSAMMENFASSUNG ===
Die KI lag in 79.7 % der Fälle richtig!
